In [5]:
# Cell 1: Install and Import Dependencies
%pip install tensorflow librosa opencv-python scikit-learn pillow

import os
import cv2
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Setup GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# -------------------------------------------------------------
# Cell 2: Image Pipeline (Dataset Preparation & Training)
# Expected structure: dataset/images/real/ and dataset/images/fake/
# -------------------------------------------------------------

IMG_SIZE = 128

def load_image_dataset(data_dir):
    X, y = [], []
    for label_str, label_val in [('real', 0), ('fake', 1)]:
        dir_path = os.path.join(data_dir, label_str)
        if not os.path.exists(dir_path): continue
        for img_name in os.listdir(dir_path):
            try:
                img_path = os.path.join(dir_path, img_name)
                img = cv2.imread(img_path)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img / 255.0  # Normalize
                X.append(img)
                y.append(label_val)
            except Exception as e:
                pass
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

# Build Image CNN
def build_image_model():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("Training Image Model...")
X_img, y_img = load_image_dataset('dataset/images')
img_train_X, img_test_X, img_train_y, img_test_y = train_test_split(X_img, y_img, test_size=0.2)
image_model = build_image_model()
image_model.fit(img_train_X, img_train_y, validation_data=(img_test_X, img_test_y), epochs=10, batch_size=32)
image_model.save('deepfake_image_model.h5')

# -------------------------------------------------------------
# Cell 3: Audio Pipeline (Dataset Preparation & Training)
# Converts raw audio into Mel-Spectrogram 2D Images
# Expected structure: dataset/audio/real/ and dataset/audio/fake/
# -------------------------------------------------------------

AUDIO_SHAPE = (128, 128, 1)

def audio_to_spectrogram(file_path, max_pad_len=128):
    try:
        y, sr = librosa.load(file_path, duration=3.0, sr=16000)
        spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_spec = librosa.power_to_db(spectrogram, ref=np.max)
        
        # Pad or truncate horizontally to maintain fixed width
        if log_spec.shape[1] < max_pad_len:
            pad_width = max_pad_len - log_spec.shape[1]
            log_spec = np.pad(log_spec, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            log_spec = log_spec[:, :max_pad_len]
            
        # Min-max normalization
        log_spec = (log_spec - np.min(log_spec)) / (np.max(log_spec) - np.min(log_spec) + 1e-6)
        return np.expand_dims(log_spec, axis=-1)
    except Exception as e:
        return None

def load_audio_dataset(data_dir):
    X, y = [], []
    for label_str, label_val in [('real', 0), ('fake', 1)]:
        dir_path = os.path.join(data_dir, label_str)
        if not os.path.exists(dir_path): continue
        for audio_name in os.listdir(dir_path):
            spec = audio_to_spectrogram(os.path.join(dir_path, audio_name))
            if spec is not None:
                X.append(spec)
                y.append(label_val)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

# Build Audio Spectrogram CNN
def build_audio_model():
    model = models.Sequential([
        layers.Input(shape=AUDIO_SHAPE),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("Training Audio Model...")
X_aud, y_aud = load_audio_dataset('dataset/audio')
aud_train_X, aud_test_X, aud_train_y, aud_test_y = train_test_split(X_aud, y_aud, test_size=0.2)
audio_model = build_audio_model()
audio_model.fit(aud_train_X, aud_train_y, validation_data=(aud_test_X, aud_test_y), epochs=10, batch_size=32)
audio_model.save('deepfake_audio_model.h5')

# -------------------------------------------------------------
# Cell 4: Video Pipeline (Dataset Preparation & Training)
# Uses Conv3D to extract spatio-temporal features
# Expected structure: dataset/video/real/ and dataset/video/fake/
# -------------------------------------------------------------

MAX_FRAMES = 15
VID_SIZE = 64

# =====================================================================
# COMPLETE SELF-CONTAINED VIDEO PIPELINE CELL FOR YOUR JUPYTER NOTEBOOK
# =====================================================================
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import train_test_split

# Global configurations matching your application structure
MAX_FRAMES = 15
VID_SIZE = 64

def extract_video_frames(video_path):
    """ Extracts frames evenly across a video clip. """
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    interval = max(1, total_frames // MAX_FRAMES)
    
    count = 0
    while cap.isOpened() and len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret: 
            break
        if count % interval == 0:
            if frame is not None and frame.size > 0:
                # FIX: Convert OpenCV BGR to RGB during training to match the app.py inference pipeline
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame_res = cv2.resize(frame_rgb, (VID_SIZE, VID_SIZE))
                frame_res = frame_res / 255.0  # Normalize pixels
                frames.append(frame_res)
        count += 1
    cap.release()
    
    # Pad zero frames if the video track is shorter than MAX_FRAMES
    while len(frames) < MAX_FRAMES:
        frames.append(np.zeros((VID_SIZE, VID_SIZE, 3)))
        
    return np.array(frames, dtype=np.float32)

def load_video_dataset(data_dir):
    """ Loops through folders and returns stacked frame arrays with label indexes. """
    X, y = [], []
    for label_str, label_val in [('real', 0), ('fake', 1)]:
        dir_path = os.path.join(data_dir, label_str)
        if not os.path.exists(dir_path): 
            print(f"Warning: Data directory route path {dir_path} not found.")
            continue
        for vid_name in os.listdir(dir_path):
            video_full_path = os.path.join(dir_path, vid_name)
            frames = extract_video_frames(video_full_path)
            X.append(frames)
            y.append(label_val)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

def build_advanced_video_model():
    """ Transfer learning model architecture using ResNet50 and LSTM. """
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(VID_SIZE, VID_SIZE, 3))
    base_model.trainable = False  # Freeze ImageNet feature extraction layers to prevent deadlocks
    
    model = models.Sequential([
        layers.TimeDistributed(base_model, input_shape=(MAX_FRAMES, VID_SIZE, VID_SIZE, 3)),
        layers.TimeDistributed(layers.GlobalAveragePooling2D()),
        
        # LSTM layer keeps track of sequential manipulation patterns over time
        layers.LSTM(64, return_sequences=False),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])
    
    opt = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# -------------------------------------------------------------
# Execution Execution Block
# -------------------------------------------------------------
print("Starting video dataset compilation pipeline...")

# Uncomment these paths when you are ready to feed your files into the model:
# X_vid, y_vid = load_video_dataset('dataset/video')
# vid_train_X, vid_test_X, vid_train_y, vid_test_y = train_test_split(X_vid, y_vid, test_size=0.2, shuffle=True)

# video_model = build_advanced_video_model()
# video_model.fit(vid_train_X, vid_train_y, validation_data=(vid_test_X, vid_test_y), epochs=15, batch_size=16)
# video_model.save('deepfake_video_model.h5')
# print("Advanced Video Model saved successfully as deepfake_video_model.h5")


Note: you may need to restart the kernel to use updated packages.
Training Image Model...
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.4583 - loss: 0.7067 - val_accuracy: 0.3333 - val_loss: 2.1160
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step - accuracy: 0.5833 - loss: 1.3389 - val_accuracy: 0.6667 - val_loss: 0.6643
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 614ms/step - accuracy: 0.4583 - loss: 0.9050 - val_accuracy: 0.3333 - val_loss: 0.7388
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 557ms/step - accuracy: 0.5000 - loss: 0.7530 - val_accuracy: 0.3333 - val_loss: 0.9192
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step - accuracy: 0.5000 - loss: 0.6966 - val_accuracy: 0.3333 - val_loss: 0.8469
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 599ms/step - accuracy: 0.7083 - loss: 0.5649 - val_accuracy: 0.3333 - val_loss: 0.8560
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 685ms/step - accuracy: 0.7500 - loss: 0.4890 - val_accuracy: 0.3333 - val_loss: 0.7199
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━

Training Audio Model...
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6761 - val_accuracy: 0.5000 - val_loss: 0.7459
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.3750 - loss: 0.8837 - val_accuracy: 0.5000 - val_loss: 1.1789
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.3750 - loss: 1.5796 - val_accuracy: 0.5000 - val_loss: 0.7486
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.6250 - loss: 0.7040 - val_accuracy: 0.5000 - val_loss: 0.8101
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.6250 - loss: 0.7832 - val_accuracy: 0.5000 - val_loss: 0.7891
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - accuracy: 0.7500 - loss: 0.5669 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - accuracy: 0.5000 - loss: 0.5692 - val_accuracy: 1.0000 - val_loss: 0.6443
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step - accuracy: 0.5000 - loss: 0.5982 - val_accurac

Starting video dataset compilation pipeline...


In [ ]:
import os
import cv2
import gradio as gr
import librosa
import numpy as np
import tensorflow as tf
import subprocess

# Load saved models with error fallback handling
try:
    image_model = tf.keras.models.load_model('deepfake_image_model.h5')
    audio_model = tf.keras.models.load_model('deepfake_audio_model.h5')
    video_model = tf.keras.models.load_model('deepfake_video_model.h5')
    print("All deepfake detection models loaded successfully.")
except Exception as e:
    print(f"Warning: Model file loading failed. Using mock logic for testing: {str(e)}")
    image_model = audio_model = video_model = None

# Constants matching training script configurations
IMG_SIZE = 128
VID_SIZE = 64
MAX_FRAMES = 15

def get_prediction_label(prob):
    """
    Standardizes label output. 
    Assumes training labels were: 0 for REAL, 1 for FAKE.
    """
    if prob < 0.5:
        confidence = (1.0 - prob) * 100
        return f"🟢 REAL (Confidence: {confidence:.2f}%)"
    else:
        confidence = prob * 100
        return f"🔴 FAKE (Confidence: {confidence:.2f}%)"

def reencode_video_to_h264(input_path):
    """
    Converts input video to browser-compatible H.264 MP4 using FFmpeg.
    This fixes the 'video not displaying/playing' error in Gradio.
    """
    output_path = os.path.join(os.path.dirname(input_path), "compatible_output.mp4")
    
    # FFmpeg command to force H264 baseline profile for web browser playback
    command = [
        'ffmpeg', '-y', '-i', input_path, 
        '-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
        '-profile:v', 'baseline', '-level', '3.0', 
        '-an', output_path
    ]
    try:
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return output_path
    except Exception:
        # Fallback to original path if ffmpeg is missing from environment variables
        print("Warning: FFmpeg re-encoding failed. Ensure FFmpeg is installed and added to PATH.")
        return input_path

def process_video_and_predict(video_file):
    """
    Handles internal video conversion for playback display and triggers inference.
    """
    if video_file is None:
        return None, "No Video Sequence Provided"
        
    # Get string file path
    video_path = str(video_file)
    
    # 1. FIX PLAYBACK: Generate a browser compatible re-encoded video instance
    web_compatible_video = reencode_video_to_h264(video_path)
    
    # 2. FIX CLASSIFICATION: Extract real video frames using OpenCV stream context
    if video_model is None: 
        return web_compatible_video, "🟢 REAL (Mock Mode - Model Not Loaded)"
        
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return web_compatible_video, "❌ Error: Unable to open video file codec stream."
        
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0:
        cap.release()
        return web_compatible_video, "❌ Error: Video file contains unreadable or empty frame tracks."
        
    interval = max(1, total_frames // MAX_FRAMES)
    count = 0
    valid_frames_read = 0
    
    while cap.isOpened() and len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret: 
            break
        if count % interval == 0:
            if frame is not None and frame.size > 0:
                frame_res = cv2.resize(frame, (VID_SIZE, VID_SIZE)) / 255.0
                frames.append(frame_res)
                valid_frames_read += 1
        count += 1
    cap.release()
    
    # Handle files with unreadable codecs or zero structural array dimensions
    if valid_frames_read == 0:
        return web_compatible_video, "❌ Error: Could not decode video frames. Try a different format (like .mp4)."
        
    # Structural padding for the model array dimensions
    while len(frames) < MAX_FRAMES:
        frames.append(np.zeros((VID_SIZE, VID_SIZE, 3)))
        
    video_input = np.expand_dims(np.array(frames, dtype=np.float32), axis=0)
    prob = float(video_model.predict(video_input, verbose=0))
    
    analysis_result = get_prediction_label(prob)
    return web_compatible_video, analysis_result

def predict_image(img):
    if img is None: 
        return "No Image Provided"
    if image_model is None: 
        return "🟢 REAL (Mock Mode - Model Not Loaded)"
        
    img_res = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
    img_input = np.expand_dims(img_res, axis=0)
    prob = float(image_model.predict(img_input, verbose=0))
    return get_prediction_label(prob)

def predict_audio(audio_path):
    if audio_path is None: 
        return "No Audio Provided"
    if audio_model is None: 
        return "🟢 REAL (Mock Mode - Model Not Loaded)"
        
    try:
        y, sr = librosa.load(audio_path, duration=3.0, sr=16000)
        spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_spec = librosa.power_to_db(spectrogram, ref=np.max)
        
        if log_spec.shape < 128:
            pad_width = 128 - log_spec.shape
            log_spec = np.pad(log_spec, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            log_spec = log_spec[:, :128]
            
        log_spec = (log_spec - np.min(log_spec)) / (np.max(log_spec) - np.min(log_spec) + 1e-6)
        audio_input = np.expand_dims(np.expand_dims(log_spec, axis=0), axis=-1)
        prob = float(audio_model.predict(audio_input, verbose=0))
        return get_prediction_label(prob)
    except Exception as e:
        return f"Error processing audio: {str(e)}"


# Gradio Interface Deployment Build
with gr.Blocks(title="Unified Real-Time Deepfake Detector") as demo:
    gr.Markdown("# 🛡️ Unified Real-Time Deepfake Detector")
    gr.Markdown("Identify synthetic deepfakes across image, audio, and video platforms.")
    
    with gr.Tab("📸 Image Deepfake Detector"):
        img_input = gr.Image(type="numpy")
        img_output = gr.Textbox(label="Analysis Result")
        img_btn = gr.Button("Scan Image")
        img_btn.click(predict_image, inputs=img_input, outputs=img_output)
        
    with gr.Tab("🎵 Audio Deepfake Detector"):
        audio_input = gr.Audio(type="filepath")
        audio_output = gr.Textbox(label="Analysis Result")
        audio_btn = gr.Button("Scan Audio File")
        audio_btn.click(predict_audio, inputs=audio_input, outputs=audio_output)
        
    with gr.Tab("🎥 Video Deepfake Detector"):
        video_input = gr.Video(label="Upload Video File")
        video_output = gr.Textbox(label="Analysis Result")
        video_btn = gr.Button("Scan Video Sequence")
        
        video_btn.click(
            process_video_and_predict, 
            inputs=video_input, 
            outputs=[video_input, video_output]
        )

if __name__ == "__main__":
    demo.launch(server_name="127.0.0.1", server_port=7867, share=False)


All deepfake detection models loaded successfully.
* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


C:\Users\user\AppData\Local\Temp\ipykernel_27640\2630034154.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prob = float(video_model.predict(video_input, verbose=0))
ERROR:asyncio:Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python310\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\user\AppData\Local\Programs\Python\Python310\lib\asyncio\proactor_events.py", line 162, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
ERROR:asyncio:Exception in

C:\Users\user\AppData\Local\Temp\ipykernel_27640\2630034154.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prob = float(video_model.predict(video_input, verbose=0))
